# Module 5.1: Deploy the Booking Agent to AgentCore Runtime

**Purpose:** Package the Module 3.1 booking agent in a container and deploy it to :link[Amazon Bedrock AgentCore Runtime]{href="https://aws.amazon.com/bedrock/agentcore/" external=true}.

**Overview**

- **AgentCore Runtime:** Runs the agent container as a managed AWS service.
- **Build context:** Contains the agent code and every file Docker can copy into the image.
- **Execution role:** Grants the Runtime access to its image, telemetry services, workload identity, and Bedrock when IAM authenticates the model call.
- **Entry point:** Reads the prompt and request ID, runs the agent, and returns the response with structured tool results.
- **Smoke tests:** Check retrieval, grounded answers, safe refusals, and reservation writes.

Module 3.1 runs the agent inside the JupyterLab kernel. Module 5.1 moves the same agent into a managed container.

| Module 3.1 | Module 5.1 |
|---|---|
| Runs in the notebook kernel | Runs in a container started by AgentCore |
| Reads the Neo4j password from the notebook environment | Receives the Neo4j password as a Runtime environment variable |
| Accepts calls from the notebook | Accepts authorized `InvokeAgentRuntime` calls |
| Keeps conversation state in the kernel | Uses a caller-provided session ID for each conversation |

The deployed agent keeps the Module 3.1 retrieval code, grounding instructions, and reservation command. It registers two read tools and one write tool. Each tool connects directly to Neo4j. The reservation command checks the guest limit and writes the reservation in one transaction.

:::alert{type="warning" header="AWS resources created"}
This notebook creates one IAM execution role, one ECR repository, one CodeBuild project, and one AgentCore Runtime. The build takes three to five minutes. Cleanup is manual. Remove these resources when you finish.
:::

In [ ]:
import os
import sys
from pathlib import Path

# The shared workshop/ package lives in notebooks/. These lines find that
# directory and put it on the import path. Everything else this notebook
# needs to start is in workshop/bootstrap.py.
_here = Path.cwd().resolve()
_named = os.environ.get("WORKSHOP_NOTEBOOKS_DIR") or _here
_starts = (Path(_named).expanduser().resolve(), _here, _here / "notebooks")
for _candidate in (*_starts, *_here.parents):
    if (_candidate / "workshop" / "bootstrap.py").is_file():
        sys.path.insert(0, str(_candidate))
        break

from workshop.bootstrap import start_module

NOTEBOOKS_ROOT, REPO_ROOT, MODULE_DIR = start_module("05-agentcore-deploy")
print(f"Workshop root: {REPO_ROOT}")

## 1. Check the deployment settings

Run the next cell before you create AWS resources. It checks your AWS credentials and the four Neo4j values that the Runtime needs.

The launch passes these Neo4j values to the container as environment variables. A failed check causes the later deployment cells to skip.

In [ ]:
import json
import shutil
import subprocess
import tempfile
import uuid
from datetime import date, timedelta
from pathlib import Path

import boto3

from workshop.aws_region import configure_aws_region
from workshop.bedrock_providers import default_model_id

# This local helper reports the Docker error behind a CodeBuild failure.
# The Runtime image does not need it.
from deploy_diagnostics import report_build_failure

REGION = configure_aws_region()
MODEL_ID = default_model_id()

# Derive every deployment name from the Runtime name. Cleanup uses the same
# rules to find these resources. Runtime names allow letters, digits, and
# underscores, so this name uses no hyphen.
RUNTIME_NAME = "GraphRagBookingAgent"
ROLE_NAME = "workshop-graphrag-runtime-role"
ECR_REPO = "workshop-graphrag-booking-agent"
CB_PROJECT = f"bedrock-agentcore-{RUNTIME_NAME.lower()}-builder"

# Cleanup selects resources with this exact tag. Each AWS API uses a different
# tag data shape.
WORKSHOP_TAG_KEY = "WorkshopResource"
WORKSHOP_TAG_VALUE = "graphrag-with-neo4j"
WORKSHOP_TAGS_MAP = {WORKSHOP_TAG_KEY: WORKSHOP_TAG_VALUE}                          # agentcore
WORKSHOP_TAGS_KV = [{"Key": WORKSHOP_TAG_KEY, "Value": WORKSHOP_TAG_VALUE}]         # ecr, iam
WORKSHOP_TAGS_KV_LOWER = [{"key": WORKSHOP_TAG_KEY, "value": WORKSHOP_TAG_VALUE}]   # codebuild

# Forward these Neo4j values to the container at launch.
NEO4J_ENV = ("NEO4J_URI", "NEO4J_USERNAME", "NEO4J_PASSWORD", "NEO4J_DATABASE")
NEO4J_VALUES = {name: os.getenv(name, "").strip() for name in NEO4J_ENV}

AWS_READY = boto3.Session().get_credentials() is not None
missing = [name for name, value in NEO4J_VALUES.items() if not value]
DEPLOY_READY = AWS_READY and not missing

print(f"region:          {REGION}")
print(f"model:           {MODEL_ID}")
print(f"runtime name:    {RUNTIME_NAME}")
print(f"execution role:  {ROLE_NAME}")
print(f"AWS credentials: {'found' if AWS_READY else 'NOT FOUND'}")
print(f"Neo4j values:    {'all four present' if not missing else 'missing ' + ', '.join(missing)}")

if not DEPLOY_READY:
    print("\nNot ready to deploy. Every live cell below will skip.")
    if missing:
        print(f"  Add to {REPO_ROOT / 'CONFIG.txt'}: {', '.join(missing)}")
    if not AWS_READY:
        print("  Configure AWS credentials before re-running this cell.")
else:
    print("\nReady to deploy.")

## 2. Prepare the Docker build context

Run the next cell to stage the files used by the container build. Docker can copy only files inside its build context.

The agent needs two dependencies stored outside `runtime_app/`. The cell builds the shared `workshop/` package as a wheel and copies `reservation_command.py` from Module 3.1. Each run replaces the staged copies. The original files remain the source of truth.

In [ ]:
DEPLOY_DIR = MODULE_DIR / "runtime_app"
if not DEPLOY_DIR.is_dir():
    raise FileNotFoundError(
        "runtime_app/ not found. Run this notebook from 05-agentcore-deploy/."
    )

PACKAGE_SRC = NOTEBOOKS_ROOT / "workshop"
COMMAND_SRC = NOTEBOOKS_ROOT / "03-grounded-booking-agent" / "reservation_command.py"

# Stage the command on every run. This local copy lets participants inspect the
# build context even when missing AWS credentials cause deployment to skip.
staged_command = Path(shutil.copy2(COMMAND_SRC, DEPLOY_DIR / "reservation_command.py"))


# Vocareum git identity: The packaged student notebook replaces this block
# through the BUILD_INFO.txt write with the pinned starter-code commit. Student
# workspaces contain copied files and have no .git directory. Keep this comment
# directly above the block.
def git_output(*args: str) -> str | None:
    """Return output from one git command when git metadata is available.

    The lab contains copied files instead of a git clone. Return None when git
    or its metadata is unavailable. Missing provenance never blocks deployment.
    """
    try:
        completed = subprocess.run(
            ["git", *args],
            cwd=REPO_ROOT,
            capture_output=True,
            text=True,
            check=True,
        )
    except (subprocess.CalledProcessError, FileNotFoundError):
        return None
    return completed.stdout.strip()


git_commit = git_output("rev-parse", "HEAD")
git_status = git_output("status", "--porcelain")
git_dirty = None if git_commit is None or git_status is None else bool(git_status)

# Store build provenance inside the image. Notebook output stays outside the
# image, while BUILD_INFO.txt travels with every rebuilt image.
(DEPLOY_DIR / "BUILD_INFO.txt").write_text(
    f"commit={git_commit or 'unknown'}\n"
    f"dirty={'unknown' if git_dirty is None else git_dirty}\n"
)

# Build the wheel in a temporary directory. Replace the staged wheel only after
# pip succeeds so a failed build leaves the previous build context usable.
with tempfile.TemporaryDirectory() as scratch:
    subprocess.run(
        [
            sys.executable, "-m", "pip", "wheel", "--no-deps",
            "--wheel-dir", scratch, str(PACKAGE_SRC),
        ],
        check=True,
    )
    fresh = sorted(Path(scratch).glob("workshop-*.whl"))
    if len(fresh) != 1:
        raise RuntimeError(f"pip built {len(fresh)} workshop wheels, expected 1")

    # Remove old wheels after the fresh build succeeds. This keeps the
    # Dockerfile's `workshop-*.whl` pattern matched to exactly one file.
    for old_wheel in DEPLOY_DIR.glob("workshop-*.whl"):
        old_wheel.unlink()
    staged_wheel = Path(shutil.copy2(fresh[0], DEPLOY_DIR / fresh[0].name))

if git_commit is None:
    print("\nBuild commit: unknown, because this tree carries no git metadata.")
else:
    print(f"\nBuild commit: {git_commit}{'  (DIRTY TREE)' if git_dirty else ''}")
    if git_dirty:
        print(
            "WARNING: uncommitted changes are present. This image will not trace to a\n"
            "git ref. Commit before deploying if this build needs to be reproducible."
        )

print(f"Staged {staged_wheel.relative_to(MODULE_DIR)}")
print(f"Staged {staged_command.relative_to(MODULE_DIR)}")
print("\nBuild context now contains:")
for path in sorted(DEPLOY_DIR.iterdir()):
    marker = "/" if path.is_dir() else ""
    print(f"  {path.name}{marker}")

## 3. Create the Runtime execution role

Run the next cell to create or update the IAM role that AgentCore assumes.

- **Image access:** Pulls the container image from Amazon ECR.
- **Telemetry:** Writes logs, traces, and metrics.
- **Workload identity:** Gets AgentCore workload access tokens.
- **Bedrock access:** Invokes the model when the container uses IAM authentication.
- **Neo4j access:** Uses the connection values passed to the container in Step 4.

The workshop uses a cross-region inference profile. Bedrock can route one request to several foundation model ARNs. The policy includes `foundation-model/*` and the account inference profile ARN so every route is allowed.

Step 4 supports two Bedrock authentication paths. A configured `AWS_BEARER_TOKEN_BEDROCK` value uses the API key. An empty value causes Step 4 to omit the variable, so the container uses the execution role. The execution role remains required for image access, telemetry, and workload identity on both paths.

In [ ]:
ROLE_ARN = ""

if not DEPLOY_READY:
    print("Skipping role creation: see Step 1.")
else:
    iam = boto3.client("iam")
    account_id = boto3.client("sts", region_name=REGION).get_caller_identity()["Account"]
    runtime_logs = f"arn:aws:logs:{REGION}:{account_id}:log-group:/aws/bedrock-agentcore/runtimes/"

    trust_policy = {
        "Version": "2012-10-17",
        "Statement": [
            {
                "Sid": "AssumeRolePolicy",
                "Effect": "Allow",
                "Principal": {"Service": "bedrock-agentcore.amazonaws.com"},
                "Action": "sts:AssumeRole",
                # Limit role assumption to AgentCore in this AWS account. These
                # conditions prevent a confused-deputy request from another account.
                "Condition": {
                    "StringEquals": {"aws:SourceAccount": account_id},
                    "ArnLike": {
                        "aws:SourceArn": f"arn:aws:bedrock-agentcore:{REGION}:{account_id}:*"
                    },
                },
            }
        ],
    }

    inline_policy = {
        "Version": "2012-10-17",
        "Statement": [
            {
                "Sid": "ECRImageAccess",
                "Effect": "Allow",
                "Action": ["ecr:BatchGetImage", "ecr:GetDownloadUrlForLayer"],
                "Resource": f"arn:aws:ecr:{REGION}:{account_id}:repository/*",
            },
            {
                "Sid": "ECRTokenAccess",
                "Effect": "Allow",
                "Action": ["ecr:GetAuthorizationToken"],
                "Resource": "*",
            },
            {
                "Sid": "RuntimeLogs",
                "Effect": "Allow",
                "Action": [
                    "logs:CreateLogGroup",
                    "logs:CreateLogStream",
                    "logs:PutLogEvents",
                    "logs:DescribeLogStreams",
                ],
                "Resource": f"{runtime_logs}*",
            },
            {
                "Sid": "DescribeLogGroups",
                "Effect": "Allow",
                "Action": ["logs:DescribeLogGroups"],
                "Resource": f"arn:aws:logs:{REGION}:{account_id}:log-group:*",
            },
            {
                "Sid": "XRay",
                "Effect": "Allow",
                "Action": [
                    "xray:PutTraceSegments",
                    "xray:PutTelemetryRecords",
                    "xray:GetSamplingRules",
                    "xray:GetSamplingTargets",
                ],
                "Resource": "*",
            },
            {
                "Sid": "CloudWatchMetrics",
                "Effect": "Allow",
                "Action": "cloudwatch:PutMetricData",
                "Resource": "*",
                "Condition": {
                    "StringEquals": {"cloudwatch:namespace": "bedrock-agentcore"}
                },
            },
            {
                "Sid": "GetAgentAccessToken",
                "Effect": "Allow",
                "Action": [
                    "bedrock-agentcore:GetWorkloadAccessToken",
                    "bedrock-agentcore:GetWorkloadAccessTokenForJWT",
                    "bedrock-agentcore:GetWorkloadAccessTokenForUserId",
                ],
                "Resource": [
                    f"arn:aws:bedrock-agentcore:{REGION}:{account_id}:workload-identity-directory/default",
                    f"arn:aws:bedrock-agentcore:{REGION}:{account_id}:workload-identity-directory/default/workload-identity/*",
                ],
            },
            {
                "Sid": "BedrockModelInvocation",
                "Effect": "Allow",
                "Action": ["bedrock:InvokeModel", "bedrock:InvokeModelWithResponseStream"],
                "Resource": [
                    "arn:aws:bedrock:*::foundation-model/*",
                    f"arn:aws:bedrock:{REGION}:{account_id}:inference-profile/*",
                ],
            },
        ],
    }

    # Create the role once. Refresh its trust policy on later runs so an existing
    # Runtime can keep using the same role.
    try:
        iam.create_role(
            RoleName=ROLE_NAME,
            AssumeRolePolicyDocument=json.dumps(trust_policy),
            Description="AgentCore Runtime execution role for the GraphRAG workshop",
            Tags=WORKSHOP_TAGS_KV,
        )
        print(f"Created role: {ROLE_NAME}")
    except iam.exceptions.EntityAlreadyExistsException:
        iam.update_assume_role_policy(
            RoleName=ROLE_NAME, PolicyDocument=json.dumps(trust_policy)
        )
        print(f"Role already exists, trust policy refreshed: {ROLE_NAME}")

    iam.put_role_policy(
        RoleName=ROLE_NAME,
        PolicyName="graphrag-runtime-policy",
        PolicyDocument=json.dumps(inline_policy),
    )
    ROLE_ARN = iam.get_role(RoleName=ROLE_NAME)["Role"]["Arn"]
    print(f"Execution role: {ROLE_ARN}")

## 4. Build the image and start the Runtime

Run the next cell to deploy the agent. The launch takes three to five minutes.

1. Check that Step 2 staged every required file.
2. Create or reuse the ECR repository.
3. Configure an ARM64 container because AgentCore Runtime uses that processor architecture.
4. Ask CodeBuild to build and push the image.
5. Create or update the AgentCore Runtime.

The cell changes to `runtime_app/` before launch. This directory contains the intended `Dockerfile` and build context.

In [ ]:
RUNTIME_ARN = ""
RUNTIME_ID = None

if not DEPLOY_READY:
    print("Skipping launch: see Step 1.")
else:
    # Check every staged file before CodeBuild receives the archive. A missing
    # wheel can stay hidden when Docker reuses a cached COPY step. This check
    # names missing files before the remote build starts.
    required = (
        "Dockerfile",
        "agent_requirements.txt",
        "booking_agent.py",
        "reservation_command.py",
        "BUILD_INFO.txt",
    )
    missing_files = [name for name in required if not (DEPLOY_DIR / name).is_file()]
    staged_wheels = sorted(DEPLOY_DIR.glob("workshop-*.whl"))
    if missing_files or len(staged_wheels) != 1:
        raise RuntimeError(
            "The build context is incomplete, so Step 2 did not finish. Re-run it "
            "and fix what it reports before launching.\n"
            f"  missing files:   {', '.join(missing_files) or 'none'}\n"
            f"  workshop wheels: {len(staged_wheels)} (expected exactly 1)"
        )

    from bedrock_agentcore_starter_toolkit import Runtime

    if Path.cwd() != DEPLOY_DIR:
        os.chdir(DEPLOY_DIR)
    print(f"Build context: {Path.cwd()}")

    # Remove the local toolkit config from a previous run. Its stale Runtime ID
    # could send this launch to a Runtime that has already been removed.
    local_config = Path.cwd() / ".bedrock_agentcore.yaml"
    if local_config.exists():
        local_config.unlink()
        print(f"Removed stale toolkit config: {local_config.name}")

    # Create the repository with the workshop prefix allowed by the participant
    # IAM policy. The toolkit's generated name uses a different prefix and would
    # block the image push for a participant role.
    ecr_setup = boto3.client("ecr", region_name=REGION)
    try:
        repo = ecr_setup.create_repository(repositoryName=ECR_REPO)["repository"]
        print(f"Created ECR repository: {ECR_REPO}")
    except ecr_setup.exceptions.RepositoryAlreadyExistsException:
        repo = ecr_setup.describe_repositories(repositoryNames=[ECR_REPO])["repositories"][0]
        print(f"Reusing ECR repository: {ECR_REPO}")
    ECR_URI = repo["repositoryUri"]
    print(f"ECR URI:       {ECR_URI}")

    agent_runtime = Runtime()
    agent_runtime.configure(
        entrypoint="booking_agent.py",
        execution_role=ROLE_ARN,
        ecr_repository=ECR_URI,
        auto_create_ecr=False,
        requirements_file="agent_requirements.txt",
        region=REGION,
        agent_name=RUNTIME_NAME,
        deployment_type="container",
        non_interactive=True,
    )

    # Forward the Bedrock API key only when it has a value. botocore selects
    # bearer authentication when this variable is present. An empty value sends
    # an invalid bearer header and prevents the SigV4 fallback.
    bedrock_api_key = os.environ.get("AWS_BEARER_TOKEN_BEDROCK", "").strip()
    bedrock_key_env = (
        {"AWS_BEARER_TOKEN_BEDROCK": bedrock_api_key} if bedrock_api_key else {}
    )
    if bedrock_api_key:
        key_tail = bedrock_api_key[-4:]
        print(
            f"Bedrock API key: forwarding {len(bedrock_api_key)} characters "
            f"ending in {key_tail}"
        )
    else:
        print("Bedrock API key: not set, so the container gets no bearer token")

    print("\nLaunching agent (3-5 minutes)...")
    try:
        result = agent_runtime.launch(
            auto_update_on_conflict=True,
            env_vars={
                # Set both region names. botocore reads AWS_DEFAULT_REGION, while the
                # workshop configuration uses AWS_REGION.
                "AWS_REGION": REGION,
                "AWS_DEFAULT_REGION": REGION,
                "MODEL_ID": MODEL_ID,
                **NEO4J_VALUES,
                **bedrock_key_env,
            },
        )
    except Exception:
        # Read the CodeBuild phase context and log before raising the toolkit's
        # summary error. Those records contain the underlying Docker failure.
        report_build_failure(CB_PROJECT, REGION)
        raise

    RUNTIME_ARN = result.agent_arn
    if not RUNTIME_ARN:
        raise RuntimeError("launch() returned no agent ARN; read the CodeBuild logs.")

    # The final ARN segment is the Runtime ID used in the CloudWatch log group.
    # Print both values so later test failures are easy to trace.
    RUNTIME_ID = RUNTIME_ARN.split("/")[-1]
    os.chdir(DEPLOY_DIR.parent)

    print(f"\nAgent deployed: {RUNTIME_ARN}")
    print(f"Runtime ID:     {RUNTIME_ID}")
    print(f"Log group:      /aws/bedrock-agentcore/runtimes/{RUNTIME_ID}-DEFAULT")

## 5. Tag the deployed resources

Run the next cell so the cleanup process can find every resource from this deployment.

Step 4 creates or updates the ECR repository, CodeBuild project, and Runtime. The toolkit leaves some workshop tags unset. This cell finds each resource by its exact name or ARN, applies the `WorkshopResource` tag, and verifies the Runtime tag.

In [ ]:
if not DEPLOY_READY:
    print("Skipping tagging: nothing was deployed.")
else:
    ecr_client = boto3.client("ecr", region_name=REGION)
    codebuild_client = boto3.client("codebuild", region_name=REGION)
    agentcore = boto3.client("bedrock-agentcore-control", region_name=REGION)

    try:
        repo = ecr_client.describe_repositories(repositoryNames=[ECR_REPO])["repositories"][0]
        ecr_client.tag_resource(resourceArn=repo["repositoryArn"], tags=WORKSHOP_TAGS_KV)
        print(f"Tagged ECR repository:  {ECR_REPO}")
    except ecr_client.exceptions.RepositoryNotFoundException:
        print(f"ECR repository not found (nothing to tag): {ECR_REPO}")

    # update_project replaces every tag. Merge the workshop tag with the existing
    # tags so the toolkit's tags remain.
    projects = codebuild_client.batch_get_projects(names=[CB_PROJECT])["projects"]
    if projects:
        merged = [t for t in projects[0].get("tags", []) if t.get("key") != WORKSHOP_TAG_KEY]
        codebuild_client.update_project(name=CB_PROJECT, tags=merged + WORKSHOP_TAGS_KV_LOWER)
        print(f"Tagged CodeBuild project: {CB_PROJECT}")
    else:
        print(f"CodeBuild project not found (nothing to tag): {CB_PROJECT}")

    if not RUNTIME_ARN:
        raise RuntimeError("RUNTIME_ARN is not set. Re-run the launch cell before tagging.")
    agentcore.tag_resource(resourceArn=RUNTIME_ARN, tags=WORKSHOP_TAGS_MAP)

    # Read the Runtime tags after the update. Cleanup could leave an untagged
    # Runtime running.
    runtime_tags = agentcore.list_tags_for_resource(resourceArn=RUNTIME_ARN).get("tags", {})
    if runtime_tags.get(WORKSHOP_TAG_KEY) != WORKSHOP_TAG_VALUE:
        raise RuntimeError(f"Runtime tag did not stick. Read back: {runtime_tags}")
    print(f"Tagged AgentCore Runtime: {RUNTIME_ARN}")
    print("\nAll toolkit-created resources tagged and verified.")

## 6. Check the deployed agent

Run the next setup cell, then run the six tests in order. Each call to `InvokeAgentRuntime` uses a separate session ID.

- **`grounding_results`:** Stores one bounded result for each read-tool call.
- **`grounding_result`:** States whether the retrieved evidence can answer the question.
- **Passage evidence:** Includes `hotel_ids` and `top_result`.
- **Structured evidence:** Includes `cypher`, `records`, and `row_count`.
- **`command_result`:** Stores the result from the reservation write tool.

The model reads the full passages inside the Runtime. The response returns only the small evidence fields needed by these checks. The tests verify tool routing, retrieved evidence, Neo4j decisions, and selected response text.

In [ ]:
from neo4j import GraphDatabase

from workshop.agent_tools import PASSAGE_TOOL, RECORD_TOOL
from workshop.contracts import MAX_GUESTS, OVER_LIMIT_GUESTS
from workshop.fixtures import (
    HERO_ADDRESS,
    HERO_NAME,
    HERO_RATING,
    HERO_SOURCE,
    load_manifest,
)
from workshop.hybrid_retrieval import Neo4jConfig

# Ask for the full address so the test can compare the exact recorded value.
HERO_QUESTION = (
    f"What is the full street address and guest rating of {HERO_NAME}? "
    "Quote the address exactly as it is recorded."
)
AGGREGATE_QUESTION = "What is the average guest rating of hotels in Paris?"
AVAILABILITY_QUESTION = f"Does {HERO_NAME} guarantee room availability next weekend?"

# Use one hotel ID that is absent from the graph. Comparing it with the fixture
# hotel separates a safe refusal from a failed retrieval system.
ABSENT_HOTEL_ID = "00000000-0000-4000-8000-000000000000"

# Reuse one caller-created UUID for every delivery of the same reservation.
# It serves as both the idempotency key and the log correlation ID.
REQUEST_ID = str(uuid.uuid4())

# Set the stay relative to today so the check-in date remains in the future.
CHECK_IN = (date.today() + timedelta(days=30)).isoformat()
CHECK_OUT = (date.today() + timedelta(days=32)).isoformat()

RESERVATION_QUERY = (
    "MATCH (r:ReservationRequest {request_id: $rid})-[:FOR_HOTEL]->(h:Hotel) "
    "RETURN r.status AS status, r.guests AS guests, h.hotel_id AS hotel_id, "
    "h.name AS hotel_name, toString(r.created_at) AS created_at"
)


def read_result(result, tool_name):
    """Return the latest recorded result for one read tool."""
    matches = [
        item
        for item in result.get("grounding_results") or []
        if item.get("tool_name") == tool_name
    ]
    assert matches, f"{tool_name} did not return a recorded result: {result}"
    return matches[-1]


def ask(prompt, runtime_arn, request_id=None, session_id=None):
    """Invoke one Runtime and print its response.

    Pass `runtime_arn` explicitly so every call shows its target. This prevents
    a stale module-level ARN from sending a request to the wrong Runtime.
    """
    payload = {"prompt": prompt}
    if request_id is not None:
        payload["request_id"] = request_id

    client = boto3.client("bedrock-agentcore", region_name=REGION)
    response = client.invoke_agent_runtime(
        agentRuntimeArn=runtime_arn,
        runtimeSessionId=session_id or str(uuid.uuid4()),
        payload=json.dumps(payload).encode("utf-8"),
        qualifier="DEFAULT",
    )
    result = json.loads(response["response"].read())

    print(f"Q: {prompt}\n")
    print(f"A: {result.get('response')}\n")
    print(f"tools used:        {result.get('tools_used') or 'none'}")
    print(f"grounding results: {result.get('grounding_results') or 'none'}")
    print(f"command result:    {result.get('command_result') or 'none'}")
    return result


def reservation_rows(request_id):
    """Read the reservation rows written to the graph.

    Use the stored graph rows as the authoritative result of the write.
    """
    config = Neo4jConfig.from_environment()
    driver = GraphDatabase.driver(config.uri, auth=(config.username, config.password))
    try:
        with driver.session(database=config.database) as session:
            return [record.data() for record in session.run(RESERVATION_QUERY, rid=request_id)]
    finally:
        driver.close()


if DEPLOY_READY:
    HERO_ID = load_manifest().hotels[HERO_SOURCE]
    print(f"Hero hotel_id:  {HERO_ID}")
    print(f"Request ID:     {REQUEST_ID}")
    print(f"Stay:           {CHECK_IN} to {CHECK_OUT}")

### Test 1: Retrieve exact hotel details

Run this test first to confirm that the Runtime can reach the graph and search index. It checks the fixture hotel's exact address and rating.

The expected values come from `workshop.fixtures`. This shared fixture keeps the test aligned with the workshop data. A successful result also makes the safe refusals in later tests meaningful.

In [ ]:
if not DEPLOY_READY:
    print("Skipping: nothing was deployed.")
else:
    hero_result = ask(HERO_QUESTION, RUNTIME_ARN)
    passage = read_result(hero_result, PASSAGE_TOOL)
    grounding = passage.get("grounding_result") or {}
    top = passage.get("top_result") or {}

    assert grounding.get("answerable") is True, grounding
    assert HERO_ID in (passage.get("hotel_ids") or []), passage

    # Compare every exact fixture value. A substring or non-empty check could pass
    # with a different Cairo hotel.
    assert top.get("hotel_id") == HERO_ID, top
    assert top.get("hotel_name") == HERO_NAME, top
    assert top.get("address") == HERO_ADDRESS, top
    assert top.get("guest_rating") == HERO_RATING, top

    # Check the model response after the tool result. This covers retrieval and
    # final answer generation.
    assert HERO_ADDRESS in (hero_result.get("response") or ""), hero_result.get("response")
    print(f"\nPASS: returned the recorded address {HERO_ADDRESS} and rating {HERO_RATING}.")

### Test 2: Use structured records for an average

Run this test to ask for the average rating across a city. This aggregate needs structured graph records.

The test checks that the agent selects `query_hotel_records`. It also checks that Text2Cypher returns inspectable Cypher and at least one result row.

In [ ]:
if not DEPLOY_READY:
    print("Skipping: nothing was deployed.")
else:
    aggregate_result = ask(AGGREGATE_QUESTION, RUNTIME_ARN)
    structured = read_result(aggregate_result, RECORD_TOOL)
    grounding = structured.get("grounding_result") or {}

    assert RECORD_TOOL in (aggregate_result.get("tools_used") or []), aggregate_result
    assert grounding.get("answerable") is True, grounding
    assert structured.get("cypher"), structured
    assert structured.get("row_count", 0) >= 1, structured
    print("\nPASS: aggregate question used structured records and returned a row.")

### Test 3: Reject an unknown hotel ID

Run this test with a hotel ID that is absent from the graph and retrieved evidence.

The expected result has no accepted command and no matching reservation in Neo4j. The agent can decline before it calls the reservation command.

In [ ]:
if not DEPLOY_READY:
    print("Skipping: nothing was deployed.")
else:
    absent_request_id = str(uuid.uuid4())
    absent = ask(
        f"Create a reservation request at hotel {ABSENT_HOTEL_ID} from {CHECK_IN} "
        f"to {CHECK_OUT} for 2 guests.",
        RUNTIME_ARN,
        request_id=absent_request_id,
    )
    absent_command = absent.get("command_result") or {}
    read_results = absent.get("grounding_results") or []

    assert absent_command.get("status") != "accepted", absent_command
    assert reservation_rows(absent_request_id) == [], "wrote a request for a hotel that does not exist"
    assert all(
        ABSENT_HOTEL_ID not in (item.get("hotel_ids") or [])
        for item in read_results
    ), read_results
    print("\nPASS: refused an ungrounded hotel_id, nothing written.")

### Test 4: Decline a live availability question

Run this test to ask for live room availability. The graph stores hotel facts and has no live room inventory.

The expected grounding result is `answerable: false` with `missing_fact: live_room_availability`. The result must also include the fixture hotel's exact address. That address proves that retrieval succeeded before the agent declined.

In [ ]:
if not DEPLOY_READY:
    print("Skipping: nothing was deployed.")
else:
    availability_result = ask(AVAILABILITY_QUESTION, RUNTIME_ARN)
    passage = read_result(availability_result, PASSAGE_TOOL)
    grounding = passage.get("grounding_result") or {}

    assert grounding.get("answerable") is False, grounding
    assert grounding.get("missing_fact") == "live_room_availability", grounding

    # Require the exact address to prove retrieval worked before the agent
    # declined the unsupported availability question.
    top = passage.get("top_result") or {}
    assert top.get("address") == HERO_ADDRESS, top
    print("\nPASS: abstained on availability while retrieval was demonstrably live.")

### Test 5: Reject an over-limit reservation

Run this test with a guest count above the workshop limit. `search_hotel_passages` must first return the stable `hotel_id` for the supplied hotel name.

The reservation transaction checks the guest limit before it writes. The test verifies the retrieved hotel ID, the rejection reason, and the absence of a reservation node.

In [ ]:
if not DEPLOY_READY:
    print("Skipping: nothing was deployed.")
else:
    rejected = ask(
        f"Create a reservation request at the hotel named {HERO_NAME} from "
        f"{CHECK_IN} to {CHECK_OUT} for {OVER_LIMIT_GUESTS} guests.",
        RUNTIME_ARN,
        request_id=REQUEST_ID,
    )
    passage = read_result(rejected, PASSAGE_TOOL)
    command = rejected.get("command_result") or {}

    assert HERO_ID in (passage.get("hotel_ids") or []), passage
    assert command.get("status") == "rejected", command
    assert command.get("reason_code") == "max_guests_exceeded", command
    # Confirm that search supplied the hotel ID because the prompt used only a name.
    assert command.get("hotel_id") == HERO_ID, command
    assert reservation_rows(REQUEST_ID) == [], "an over-limit request was written"
    print(f"\nPASS: rejected with reason_code={command.get('reason_code')}, no node written.")

### Test 6: Write one reservation and handle a retry

Run this test with a valid guest count. The first call creates one `ReservationRequest` linked to the fixture hotel.

The second call sends the same `request_id` from a new session. It should return `duplicate=true` with the original `created_at`. The final graph check must still find one node.

In [ ]:
if not DEPLOY_READY:
    print("Skipping: nothing was deployed.")
else:
    reservation_prompt = (
        f"Create a reservation request at the hotel named {HERO_NAME} from "
        f"{CHECK_IN} to {CHECK_OUT} for {MAX_GUESTS} guests."
    )
    accepted = ask(reservation_prompt, RUNTIME_ARN, request_id=REQUEST_ID)
    accepted_passage = read_result(accepted, PASSAGE_TOOL)
    command = accepted.get("command_result") or {}
    assert HERO_ID in (accepted_passage.get("hotel_ids") or []), accepted_passage
    assert command.get("status") == "accepted", command
    assert command.get("hotel_id") == HERO_ID, command

    rows = reservation_rows(REQUEST_ID)
    assert len(rows) == 1, rows
    assert rows[0]["hotel_id"] == HERO_ID, rows
    assert rows[0]["guests"] == MAX_GUESTS, rows
    print(f"\nGraph says: {json.dumps(rows[0], indent=2)}")

    # Use a new session ID for the replay. This proves that the idempotency key,
    # rather than conversation history, detects the duplicate.
    replay = ask(reservation_prompt, RUNTIME_ARN, request_id=REQUEST_ID)
    replay_passage = read_result(replay, PASSAGE_TOOL)
    replay_command = replay.get("command_result") or {}
    assert HERO_ID in (replay_passage.get("hotel_ids") or []), replay_passage
    assert replay_command.get("duplicate") is True, replay_command
    assert len(reservation_rows(REQUEST_ID)) == 1, "replay created a second node"
    print("\nPASS: one node written, replay returned duplicate=true, still one node.")

## 7. Read recent Runtime logs

Run the next cell to read recent events from the Runtime's CloudWatch log group. Each successful invocation writes a start line and a completion line. The completion line includes the tools used and command status.

Use the caller-provided `request_id` to connect a reservation request with its log entries.

In [ ]:
if not DEPLOY_READY or not RUNTIME_ID:
    print("Skipping logs: nothing was deployed.")
else:
    from datetime import datetime, timezone

    logs_client = boto3.client("logs", region_name=REGION)
    log_group = f"/aws/bedrock-agentcore/runtimes/{RUNTIME_ID}-DEFAULT"
    start_time = int(
        (datetime.now(timezone.utc) - timedelta(hours=2)).timestamp() * 1000
    )
    try:
        response = logs_client.filter_log_events(
            logGroupName=log_group,
            startTime=start_time,
            limit=100,
        )
    except logs_client.exceptions.ResourceNotFoundException:
        print(f"No log group found yet: {log_group}")
    else:
        events = response.get("events", [])
        print(f"Recent events from {log_group}: {len(events)}")
        for event in events:
            timestamp = datetime.fromtimestamp(
                event["timestamp"] / 1000,
                tz=timezone.utc,
            ).isoformat()
            print(f"{timestamp}  {event['message'].rstrip()}")

## Review the deployed architecture

```
InvokeAgentRuntime
        |
        v
+---------------------------+
|  AgentCore Runtime        |
|  GraphRagBookingAgent     |
|                           |
|  booking_agent.py         |
|   +- search_hotel_passages  --> Neo4j hybrid retrieval
|   +- query_hotel_records    --> read-only Text2Cypher
|   +- create_reservation     --> Neo4j write, rule enforced in-transaction
|   +- BedrockModel           --> Claude on Amazon Bedrock
+---------------------------+
```

A request follows this path. An authorized application calls `InvokeAgentRuntime`. AgentCore runs `booking_agent.py`. The agent selects a tool, reads its result, and uses Claude to write the response.

- **`InvokeAgentRuntime`:** Sends a request to the deployed agent.
- **AgentCore Runtime:** Runs the container and supplies its Neo4j environment variables.
- **GraphRagBookingAgent:** Receives the request and selects a tool.
- **`search_hotel_passages`:** Finds source text and linked hotel facts with hybrid retrieval.
- **`query_hotel_records`:** Creates and runs read-only Cypher for structured questions.
- **`create_reservation`:** Writes a request after Neo4j checks the reservation rule.
- **`BedrockModel`:** Uses Claude to understand the request and write the answer.

The write tool accepts only a `hotel_id` returned by `search_hotel_passages` during the same invocation. This rule connects every reservation to a hotel retrieved from Neo4j.

**Verified results**

- **Hotel details:** The agent returned the exact address stored in Neo4j.
- **Structured records:** The agent created read-only Cypher and returned a row.
- **Missing live data:** The agent identified the missing inventory fact and declined the availability question.
- **Reservation limit:** Neo4j rejected a request above the guest limit.
- **Idempotent retry:** The repeated request returned the original reservation and kept one graph node.

:::alert{type="info" header="Clean up resources when you finish"}
Module 6 uses different resources. Find the Module 5 resources by their `WorkshopResource` tag. Remove the Runtime, ECR repository, CodeBuild project, and execution role when you finish.
:::